# LangChain 1.x — Demo từng Method với từng bộ tham số

**Seminar môn học · Tài liệu thực hành kèm slide**

Notebook này đi qua **từng method** của LangChain, mỗi method chạy với **nhiều bộ
tham số khác nhau** để thấy rõ tham số ảnh hưởng thế nào tới kết quả.

## Cách dùng notebook

| Chế độ | Cần gì | Chạy được phần nào |
|---|---|---|
| **Offline** (mặc định) | Không cần API key | Mục 1–5, 9 (phần không gọi LLM) |
| **Online** | 1 API key bất kỳ | Toàn bộ |

Notebook tự phát hiện có API key hay không. Nếu không có, các cell gọi LLM sẽ
**bỏ qua một cách an toàn** thay vì báo lỗi đỏ.

## Mục lục

| # | Nội dung | Method chính |
|---|---|---|
| 1 | Document | `Document()` |
| 2 | Text Splitter | `split_documents()`, `split_text()` |
| 3 | Embeddings | `embed_query()`, `embed_documents()` |
| 4 | Vector Store | `add_documents()`, `similarity_search()`, `mmr` |
| 5 | Retriever | `as_retriever()`, `invoke()`, `batch()` |
| 6 | Prompt Template | `from_template()`, `from_messages()`, `partial()` |
| 7 | Chat Model | `invoke()`, `stream()`, `batch()` |
| 8 | Output Parser | `StrOutputParser`, `with_structured_output()` |
| 9 | LCEL | `\|`, `RunnableParallel`, `RunnablePassthrough` |
| 10 | RAG hoàn chỉnh | ghép tất cả |
| 11 | Agent | `create_agent()`, `@tool` |
| 12 | Bảng tổng kết tham số | — |

---

## 0. Cài đặt

Chạy cell dưới một lần. Nếu đang dùng Google Colab, bỏ dấu `#` ở dòng `!pip`.

In [1]:
# !pip install -qU langchain langchain-core langchain-text-splitters
# !pip install -qU langchain-openai        # nếu dùng OpenAI
# !pip install -qU langchain-google-genai  # nếu dùng Google Gemini (có free tier)

import langchain_core
print("langchain-core:", langchain_core.__version__)

langchain-core: 1.6.9


### 0.1. Cấu hình provider

**Ba lựa chọn**, chọn 1:

1. **Không có key** → để nguyên, notebook chạy chế độ offline.
2. **Google Gemini** (khuyến nghị cho sinh viên, có free tier) →
   lấy key tại `aistudio.google.com/apikey`, điền vào `GOOGLE_API_KEY`.
3. **OpenAI** → điền `OPENAI_API_KEY`.

In [2]:
import os, getpass

# ---- Chọn provider: "google" | "openai" | "none" ----
PROVIDER = "google"   # <-- SỬA Ở ĐÂY

if PROVIDER == "google":
    if not os.environ.get("GOOGLE_API_KEY"):
        os.environ["GOOGLE_API_KEY"] = getpass.getpass("GOOGLE_API_KEY: ")
    MODEL_NAME = "google_genai:gemini-3.5-flash-lite"
    EMBED_SPEC = ("google", "models/gemini-embedding-001")
elif PROVIDER == "openai":
    if not os.environ.get("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")
    MODEL_NAME = "openai:gpt-4o-mini"
    EMBED_SPEC = ("openai", "text-embedding-3-small")
else:
    MODEL_NAME = None
    EMBED_SPEC = ("fake", None)

ONLINE = PROVIDER in ("google", "openai")
print(f"Chế độ: {'ONLINE — ' + MODEL_NAME if ONLINE else 'OFFLINE (không gọi LLM)'}")

Chế độ: OFFLINE (không gọi LLM)


In [3]:
def skip_if_offline(func):
    '''Decorator: chỉ chạy hàm khi có API key, ngược lại in thông báo.'''
    def wrapper(*args, **kwargs):
        if not ONLINE:
            print(f"[BỎ QUA] {func.__name__} — cần API key. "
                  f"Đặt PROVIDER ở cell 0.1 để chạy.")
            return None
        return func(*args, **kwargs)
    return wrapper

def rule(title=""):
    '''In đường kẻ ngăn cách cho dễ đọc output.'''
    print("\n" + "=" * 70)
    if title:
        print(title)
        print("=" * 70)

---
## 1. `Document` — đơn vị dữ liệu cơ bản

Mọi thứ trong pipeline RAG đều đi qua `Document`. Nó chỉ có **3 thuộc tính**:

| Thuộc tính | Kiểu | Bắt buộc | Ý nghĩa |
|---|---|---|---|
| `page_content` | `str` | Có | Nội dung văn bản |
| `metadata` | `dict` | Không (mặc định `{}`) | Nguồn, số trang, tác giả... dùng để lọc |
| `id` | `str \| None` | Không | Định danh, hữu ích khi cập nhật/xóa |

**Input:** `Document(page_content=..., metadata=..., id=...)`
**Output:** một object `Document`

In [4]:
from langchain_core.documents import Document

# --- Bộ tham số 1: tối giản, chỉ có nội dung ---
d1 = Document(page_content="Sinh viên được đăng ký tối đa 24 tín chỉ mỗi học kỳ.")

# --- Bộ tham số 2: thêm metadata (dùng để lọc/trích nguồn) ---
d2 = Document(
    page_content="Sinh viên bị cảnh báo học vụ chỉ được đăng ký tối đa 14 tín chỉ.",
    metadata={"source": "quy_che.md", "dieu": 4, "chuong": "Tổ chức đào tạo"},
)

# --- Bộ tham số 3: thêm id (cần khi muốn update/delete trong vector store) ---
d3 = Document(
    page_content="Học kỳ hè cho phép đăng ký tối đa 12 tín chỉ.",
    metadata={"source": "quy_che.md", "dieu": 4},
    id="qc-dieu4-hoc-ky-he",
)

for i, d in enumerate([d1, d2, d3], 1):
    print(f"--- Document {i} ---")
    print("  page_content:", d.page_content[:60], "...")
    print("  metadata    :", d.metadata)
    print("  id          :", d.id)

--- Document 1 ---
  page_content: Sinh viên được đăng ký tối đa 24 tín chỉ mỗi học kỳ. ...
  metadata    : {}
  id          : None
--- Document 2 ---
  page_content: Sinh viên bị cảnh báo học vụ chỉ được đăng ký tối đa 14 tín  ...
  metadata    : {'source': 'quy_che.md', 'dieu': 4, 'chuong': 'Tổ chức đào tạo'}
  id          : None
--- Document 3 ---
  page_content: Học kỳ hè cho phép đăng ký tối đa 12 tín chỉ. ...
  metadata    : {'source': 'quy_che.md', 'dieu': 4}
  id          : qc-dieu4-hoc-ky-he


> **Lưu ý cho nhóm làm theo:** `metadata` là chỗ đặt mọi thứ bạn muốn lọc về sau
> (`source`, `page`, `author`, `date`...). Đặt metadata ngay từ đầu, đừng để
> đến lúc cần trích dẫn nguồn mới thêm — lúc đó phải index lại toàn bộ.

---
## 2. `RecursiveCharacterTextSplitter` — cắt văn bản thành chunk

### Tại sao phải cắt?
- Context window của model có hạn.
- Chunk quá to → nhiễu, câu trả lời loãng.
- Chunk quá nhỏ → mất ngữ cảnh, câu bị cụt.

### Thuật toán
Splitter thử cắt theo **danh sách separator ưu tiên từ trên xuống**:
`["\n\n", "\n", " ", ""]`. Cắt theo `\n\n` trước; đoạn nào vẫn > `chunk_size`
thì cắt tiếp bằng `\n`; rồi tới dấu cách; cuối cùng cắt thô từng ký tự.
Nhờ vậy nó **giữ được ranh giới ngữ nghĩa** (đoạn văn, câu) càng lâu càng tốt.

### Tham số

| Tham số | Kiểu | Mặc định | Ảnh hưởng |
|---|---|---|---|
| `chunk_size` | `int` | 4000 | Độ dài tối đa mỗi chunk (đơn vị do `length_function` quyết định) |
| `chunk_overlap` | `int` | 200 | Số ký tự lặp giữa 2 chunk liền kề, chống đứt câu |
| `separators` | `list[str]` | `["\n\n","\n"," ",""]` | Thứ tự ưu tiên điểm cắt |
| `add_start_index` | `bool` | `False` | Ghi vị trí ký tự bắt đầu vào metadata |
| `length_function` | `callable` | `len` | Đổi sang đếm token nếu cần |
| `keep_separator` | `bool` | `False` | Giữ lại separator trong chunk |

**Input:** `split_documents(list[Document])` hoặc `split_text(str)`
**Output:** `list[Document]` / `list[str]`

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

VAN_BAN = '''Điều 4. Đăng ký học phần

Mỗi học kỳ chính, sinh viên phải đăng ký tối thiểu 14 tín chỉ và tối đa 24 tín chỉ.
Sinh viên thuộc diện cảnh báo học vụ chỉ được đăng ký tối đa 14 tín chỉ trong học
kỳ kế tiếp. Trong học kỳ hè, số tín chỉ tối đa được đăng ký là 12.

Thời hạn đăng ký học phần là 2 tuần trước khi học kỳ bắt đầu. Sinh viên được phép
điều chỉnh đăng ký trong 2 tuần đầu của học kỳ mà không bị ghi nhận vào bảng điểm.

Điều 5. Rút học phần

Từ tuần thứ 3 đến hết tuần thứ 8 của học kỳ, sinh viên có thể rút học phần. Học
phần bị rút được ghi ký hiệu W trên bảng điểm, không tính vào điểm trung bình
nhưng vẫn tính học phí.'''

print(f"Độ dài văn bản gốc: {len(VAN_BAN)} ký tự")

Độ dài văn bản gốc: 629 ký tự


### 2.1. So sánh các bộ tham số `chunk_size` / `chunk_overlap`

Chạy trên **hai loại văn bản** để lộ ra một hành vi mà rất nhiều người hiểu sai:

- `VAN_BAN` — có ngắt đoạn `\n\n` rõ ràng (như quy chế, tài liệu có cấu trúc).
- `VAN_BAN_LIEN_TUC` — một khối chữ liền mạch, không ngắt đoạn (như transcript,
  OCR, chat log).

In [6]:
VAN_BAN_LIEN_TUC = (
    "Mỗi học kỳ chính, sinh viên phải đăng ký tối thiểu 14 tín chỉ và tối đa 24 "
    "tín chỉ. Sinh viên thuộc diện cảnh báo học vụ chỉ được đăng ký tối đa 14 "
    "tín chỉ trong học kỳ kế tiếp. Trong học kỳ hè, số tín chỉ tối đa được đăng "
    "ký là 12 tín chỉ. Thời hạn đăng ký học phần là 2 tuần trước khi học kỳ bắt "
    "đầu."
)

BO_THAM_SO = [
    {"chunk_size": 120, "chunk_overlap": 0},
    {"chunk_size": 120, "chunk_overlap": 40},
    {"chunk_size": 250, "chunk_overlap": 50},
    {"chunk_size": 600, "chunk_overlap": 100},
]

for ten, van_ban in [("CÓ ngắt đoạn (VAN_BAN)", VAN_BAN),
                     ("KHÔNG ngắt đoạn (VAN_BAN_LIEN_TUC)", VAN_BAN_LIEN_TUC)]:
    rule(f"{ten} — {len(van_ban)} ký tự")
    print(f"{'chunk_size':>11} {'overlap':>8} {'số chunk':>9} {'dài nhất':>9} "
          f"{'ngắn nhất':>10} {'TB':>7}")
    print("-" * 60)
    for ts in BO_THAM_SO:
        chunks = RecursiveCharacterTextSplitter(**ts).split_text(van_ban)
        dd = [len(c) for c in chunks]
        print(f"{ts['chunk_size']:>11} {ts['chunk_overlap']:>8} {len(chunks):>9} "
              f"{max(dd):>9} {min(dd):>10} {round(sum(dd)/len(dd), 1):>7}")


CÓ ngắt đoạn (VAN_BAN) — 629 ký tự
 chunk_size  overlap  số chunk  dài nhất  ngắn nhất      TB
------------------------------------------------------------
        120        0         9       101         20    68.6
        120       40         9       101         20    68.6
        250       50         4       232         24   161.2
        600      100         2       446        203   324.5

KHÔNG ngắt đoạn (VAN_BAN_LIEN_TUC) — 302 ký tự
 chunk_size  overlap  số chunk  dài nhất  ngắn nhất      TB
------------------------------------------------------------
        120        0         3       120         61   100.0
        120       40         4       120         61   103.0
        250       50         2       249        100   174.5
        600      100         1       302        302   302.0


**Đọc bảng trên cho kỹ — đây là điểm dễ hiểu sai nhất:**

1. Ở văn bản **có ngắt đoạn**, hai dòng `chunk_size=120` với `overlap=0` và
   `overlap=40` cho kết quả **y hệt nhau**. Overlap dường như không làm gì cả.

2. Ở văn bản **không ngắt đoạn**, `overlap=40` làm **tăng số chunk**.

**Vì sao?** Thuật toán chạy hai giai đoạn: (a) *tách* văn bản theo separator, rồi
(b) *gộp* các mẩu nhỏ lại cho tới sát `chunk_size`. `chunk_overlap` **chỉ tác
động ở giai đoạn gộp**. Nếu văn bản đã tự tách thành các đoạn đều nhỏ hơn
`chunk_size`, mỗi đoạn thành một chunk luôn, không có gì để gộp → overlap vô hiệu.

**Hệ quả thực tế:** đừng chỉ nhìn tham số bạn đặt, hãy **in chunk ra xem**. Rất
nhiều nhóm đặt `chunk_overlap=200` rồi yên tâm, trong khi tài liệu markdown của
họ ngắt đoạn dày đặc nên overlap chẳng bao giờ được áp dụng.

### 2.2. `chunk_overlap` thực sự làm gì? — nhìn tận mắt

Dùng văn bản liền mạch để overlap chắc chắn được kích hoạt.

In [7]:
for overlap in [0, 40]:
    sp = RecursiveCharacterTextSplitter(chunk_size=120, chunk_overlap=overlap)
    chunks = sp.split_text(VAN_BAN_LIEN_TUC)
    rule(f"chunk_overlap = {overlap}  →  {len(chunks)} chunk")
    for i, c in enumerate(chunks):
        print(f"[chunk {i}] ({len(c)} ký tự)")
        print("   ", repr(c))
    print()


chunk_overlap = 0  →  3 chunk
[chunk 0] (120 ký tự)
    'Mỗi học kỳ chính, sinh viên phải đăng ký tối thiểu 14 tín chỉ và tối đa 24 tín chỉ. Sinh viên thuộc diện cảnh báo học vụ'
[chunk 1] (119 ký tự)
    'chỉ được đăng ký tối đa 14 tín chỉ trong học kỳ kế tiếp. Trong học kỳ hè, số tín chỉ tối đa được đăng ký là 12 tín chỉ.'
[chunk 2] (61 ký tự)
    'Thời hạn đăng ký học phần là 2 tuần trước khi học kỳ bắt đầu.'


chunk_overlap = 40  →  4 chunk
[chunk 0] (120 ký tự)
    'Mỗi học kỳ chính, sinh viên phải đăng ký tối thiểu 14 tín chỉ và tối đa 24 tín chỉ. Sinh viên thuộc diện cảnh báo học vụ'
[chunk 1] (117 ký tự)
    'Sinh viên thuộc diện cảnh báo học vụ chỉ được đăng ký tối đa 14 tín chỉ trong học kỳ kế tiếp. Trong học kỳ hè, số tín'
[chunk 2] (114 ký tự)
    'học kỳ kế tiếp. Trong học kỳ hè, số tín chỉ tối đa được đăng ký là 12 tín chỉ. Thời hạn đăng ký học phần là 2 tuần'
[chunk 3] (61 ký tự)
    'Thời hạn đăng ký học phần là 2 tuần trước khi học kỳ bắt đầu.'



Với `overlap=40`, cụm *"Sinh viên thuộc diện cảnh báo học vụ"* ở **cuối chunk 0**
xuất hiện lại ở **đầu chunk 1**; tương tự, cụm *"học kỳ kế tiếp. Trong học kỳ hè"*
nằm trong chunk 1 và được lặp lại ở **đầu chunk 2**.

Đó chính là tác dụng: dù điểm cắt rơi vào giữa một ý, **luôn có một chunk chứa
trọn ý đó**. Không có overlap, câu *"Sinh viên thuộc diện cảnh báo học vụ..."* bị
xé làm đôi và **không chunk nào** trả lời được câu hỏi về diện cảnh báo học vụ.

**Quy tắc ngón tay cái:** `chunk_overlap ≈ 10–20% của chunk_size`, và luôn in vài
chunk ra kiểm tra xem overlap có thực sự được áp dụng không.

### 2.3. `separators` — đổi thứ tự ưu tiên điểm cắt

In [8]:
CAU_HINH_SEP = [
    (["\n\n", "\n", " ", ""],  "mặc định — ưu tiên đoạn văn"),
    (["\n", " ", ""],           "bỏ qua đoạn văn, cắt theo dòng"),
    ([". ", " ", ""],           "cắt theo câu"),
    ([""],                      "cắt thô từng ký tự — TỆ NHẤT"),
]

for seps, mo_ta in CAU_HINH_SEP:
    sp = RecursiveCharacterTextSplitter(
        chunk_size=150, chunk_overlap=0, separators=seps
    )
    chunks = sp.split_text(VAN_BAN)
    rule(f"separators={seps}  ({mo_ta})")
    print(f"→ {len(chunks)} chunk. Chunk đầu tiên:")
    print(repr(chunks[0]))


separators=['\n\n', '\n', ' ', '']  (mặc định — ưu tiên đoạn văn)
→ 8 chunk. Chunk đầu tiên:
'Điều 4. Đăng ký học phần'

separators=['\n', ' ', '']  (bỏ qua đoạn văn, cắt theo dòng)
→ 6 chunk. Chunk đầu tiên:
'Điều 4. Đăng ký học phần\n\nMỗi học kỳ chính, sinh viên phải đăng ký tối thiểu 14 tín chỉ và tối đa 24 tín chỉ.'

separators=['. ', ' ', '']  (cắt theo câu)
→ 7 chunk. Chunk đầu tiên:
'Điều 4'

separators=['']  (cắt thô từng ký tự — TỆ NHẤT)
→ 5 chunk. Chunk đầu tiên:
'Điều 4. Đăng ký học phần\n\nMỗi học kỳ chính, sinh viên phải đăng ký tối thiểu 14 tín chỉ và tối đa 24 tín chỉ.\nSinh viên thuộc diện cảnh báo học vụ chỉ'


Cấu hình cuối (`separators=[""]`) cho thấy điều gì xảy ra khi bỏ hết ranh giới
ngữ nghĩa: chunk bị cắt giữa chừng một từ. Đây chính là lỗi hay gặp khi nhóm tự
viết hàm cắt bằng `text[i:i+n]`.

### 2.4. `add_start_index` — truy vết vị trí gốc

In [9]:
docs_goc = [Document(page_content=VAN_BAN, metadata={"source": "quy_che.md"})]

for co_index in [False, True]:
    sp = RecursiveCharacterTextSplitter(
        chunk_size=200, chunk_overlap=40, add_start_index=co_index
    )
    splits = sp.split_documents(docs_goc)
    rule(f"add_start_index = {co_index}")
    for s in splits[:3]:
        print(" ", s.metadata)


add_start_index = False
  {'source': 'quy_che.md'}
  {'source': 'quy_che.md'}
  {'source': 'quy_che.md'}

add_start_index = True
  {'source': 'quy_che.md', 'start_index': 0}
  {'source': 'quy_che.md', 'start_index': 26}
  {'source': 'quy_che.md', 'start_index': 192}


Với `add_start_index=True`, mỗi chunk biết nó nằm ở ký tự thứ mấy của tài liệu
gốc — cực kỳ hữu ích khi muốn highlight đoạn trích dẫn trong file gốc.

Lưu ý: `split_documents()` **tự động copy metadata** của document cha sang tất cả
các chunk con.

### 2.5. Đếm theo token thay vì ký tự

Model tính tiền theo **token**, không phải ký tự. Với tiếng Việt, 1 token ≈ 2–3
ký tự (tệ hơn tiếng Anh do dấu). Nếu muốn `chunk_size` khớp với giới hạn thật
của model, đổi `length_function`.

In [10]:
try:
    import tiktoken
    enc = tiktoken.get_encoding("cl100k_base")

    def dem_token(text: str) -> int:
        return len(enc.encode(text))

    sp_token = RecursiveCharacterTextSplitter(
        chunk_size=60, chunk_overlap=10, length_function=dem_token
    )
    chunks = sp_token.split_text(VAN_BAN)
    print(f"Cắt theo TOKEN: {len(chunks)} chunk")
    for c in chunks[:2]:
        print(f"  {dem_token(c):>3} token / {len(c):>3} ký tự  "
              f"→ tỷ lệ {len(c)/dem_token(c):.2f} ký tự/token")
except ImportError:
    print("[BỎ QUA] Cần: pip install tiktoken")

[BỎ QUA] Cần: pip install tiktoken


---
## 3. Embeddings — biến văn bản thành vector

Embedding model ánh xạ một chuỗi văn bản thành vector số thực sao cho **văn bản
nghĩa gần nhau thì vector gần nhau** trong không gian đó.

### Hai method chính

| Method | Input | Output | Dùng khi |
|---|---|---|---|
| `embed_query(text)` | `str` | `list[float]` | Nhúng **câu hỏi** của người dùng |
| `embed_documents(texts)` | `list[str]` | `list[list[float]]` | Nhúng **kho tài liệu** (theo lô) |

> Một số provider nhúng query và document **khác nhau** (thêm prefix nội bộ), nên
> phải dùng đúng method, không được gọi `embed_documents([cau_hoi])[0]` thay cho
> `embed_query(cau_hoi)`.

Bản `Fake` dưới đây cho vector ngẫu nhiên **ổn định theo nội dung** — đủ để chạy
thử cơ chế vector store offline, nhưng **không có ý nghĩa ngữ nghĩa**.

In [11]:
kind, model_name = EMBED_SPEC

if kind == "openai":
    from langchain_openai import OpenAIEmbeddings
    embeddings = OpenAIEmbeddings(model=model_name)
elif kind == "google":
    from langchain_google_genai import GoogleGenerativeAIEmbeddings
    embeddings = GoogleGenerativeAIEmbeddings(model=model_name)
else:
    from langchain_core.embeddings import DeterministicFakeEmbedding
    embeddings = DeterministicFakeEmbedding(size=256)

print("Embedding đang dùng:", type(embeddings).__name__)

Embedding đang dùng: DeterministicFakeEmbedding


In [12]:
v = embeddings.embed_query("Sinh viên được đăng ký tối đa bao nhiêu tín chỉ?")
print(f"embed_query  → 1 vector, số chiều = {len(v)}")
print(f"  5 giá trị đầu: {[round(x, 4) for x in v[:5]]}")

vs = embeddings.embed_documents([
    "Sinh viên đăng ký tối đa 24 tín chỉ.",
    "Lệ phí phúc khảo là 50.000 đồng.",
    "Thời gian đào tạo tối đa là 8 năm.",
])
print(f"\nembed_documents → {len(vs)} vector, mỗi vector {len(vs[0])} chiều")

embed_query  → 1 vector, số chiều = 256
  5 giá trị đầu: [np.float64(0.1227), np.float64(0.8041), np.float64(-0.7813), np.float64(0.8078), np.float64(-1.4417)]

embed_documents → 3 vector, mỗi vector 256 chiều


### 3.1. Cosine similarity — thước đo "gần nhau"

$$\text{cosine}(\vec{a},\vec{b}) = \frac{\vec{a}\cdot\vec{b}}{\|\vec{a}\|\,\|\vec{b}\|}$$

Giá trị chạy từ −1 đến 1. Càng gần **1** thì hai văn bản càng giống nghĩa.

In [13]:
import math

def cosine(a, b):
    tich = sum(x * y for x, y in zip(a, b))
    na = math.sqrt(sum(x * x for x in a))
    nb = math.sqrt(sum(y * y for y in b))
    return tich / (na * nb)

cau_hoi = "Một học kỳ được đăng ký nhiều nhất bao nhiêu tín chỉ?"
ung_vien = [
    "Mỗi học kỳ chính sinh viên đăng ký tối đa 24 tín chỉ.",   # đúng chủ đề
    "Sinh viên bị cảnh báo học vụ đăng ký tối đa 14 tín chỉ.", # gần chủ đề
    "Lệ phí phúc khảo là 50.000 đồng một học phần.",           # khác chủ đề
    "Hôm nay Sài Gòn trời mưa rất to.",                        # hoàn toàn lạc
]

vq = embeddings.embed_query(cau_hoi)
vd = embeddings.embed_documents(ung_vien)

print(f"Câu hỏi: {cau_hoi}\n")
diem = sorted(zip(ung_vien, [cosine(vq, x) for x in vd]),
              key=lambda t: -t[1])
for text, sim in diem:
    thanh = "#" * int(max(sim, 0) * 40)
    print(f"{sim:+.4f} |{thanh:<40}| {text[:50]}")

if not ONLINE:
    print("\n[!] Đang dùng FakeEmbedding → điểm số VÔ NGHĨA, chỉ minh họa cơ chế.")
    print("    Đặt PROVIDER ở cell 0.1 để thấy thứ tự đúng theo ngữ nghĩa.")

Câu hỏi: Một học kỳ được đăng ký nhiều nhất bao nhiêu tín chỉ?

+0.0470 |#                                       | Mỗi học kỳ chính sinh viên đăng ký tối đa 24 tín c
+0.0441 |#                                       | Sinh viên bị cảnh báo học vụ đăng ký tối đa 14 tín
+0.0173 |                                        | Hôm nay Sài Gòn trời mưa rất to.
-0.0794 |                                        | Lệ phí phúc khảo là 50.000 đồng một học phần.

[!] Đang dùng FakeEmbedding → điểm số VÔ NGHĨA, chỉ minh họa cơ chế.
    Đặt PROVIDER ở cell 0.1 để thấy thứ tự đúng theo ngữ nghĩa.


---
## 4. Vector Store — lưu và tìm kiếm vector

`InMemoryVectorStore` lưu toàn bộ trong RAM, mất khi tắt notebook. Đủ cho demo và
prototype. Khi lên production thì đổi sang Chroma / Qdrant / PGVector — **API y
hệt nhau**, chỉ đổi dòng khởi tạo.

### Các method

| Method | Input | Output |
|---|---|---|
| `add_documents(documents)` | `list[Document]` | `list[str]` (danh sách id) |
| `similarity_search(query, k)` | `str`, `int` | `list[Document]` |
| `similarity_search_with_score(query, k)` | `str`, `int` | `list[tuple[Document, float]]` |
| `similarity_search_by_vector(embedding, k)` | `list[float]`, `int` | `list[Document]` |
| `max_marginal_relevance_search(query, k, fetch_k, lambda_mult)` | | `list[Document]` |
| `as_retriever(...)` | | `VectorStoreRetriever` |

In [14]:
from langchain_core.vectorstores import InMemoryVectorStore

KHO_TAI_LIEU = [
    Document(page_content="Mỗi học kỳ chính, sinh viên phải đăng ký tối thiểu 14 "
             "tín chỉ và tối đa 24 tín chỉ.",
             metadata={"source": "quy_che", "dieu": 4}),
    Document(page_content="Sinh viên thuộc diện cảnh báo học vụ chỉ được đăng ký "
             "tối đa 14 tín chỉ trong học kỳ kế tiếp.",
             metadata={"source": "quy_che", "dieu": 4}),
    Document(page_content="Trong học kỳ hè, số tín chỉ tối đa được đăng ký là 12.",
             metadata={"source": "quy_che", "dieu": 4}),
    Document(page_content="Từ tuần thứ 3 đến hết tuần thứ 8, sinh viên có thể rút "
             "học phần và được ghi ký hiệu W trên bảng điểm.",
             metadata={"source": "quy_che", "dieu": 5}),
    Document(page_content="Sinh viên vắng mặt quá 20% số tiết sẽ bị cấm thi và "
             "nhận điểm F cho học phần đó.",
             metadata={"source": "quy_che", "dieu": 7}),
    Document(page_content="Đơn phúc khảo nộp tại phòng Khảo thí trong vòng 10 ngày "
             "kể từ ngày công bố điểm. Lệ phí là 50.000 đồng một học phần.",
             metadata={"source": "faq", "muc": "B2"}),
    Document(page_content="Sinh viên đã tích lũy tối thiểu 90 tín chỉ được đăng ký "
             "thực tập doanh nghiệp, thời lượng 8 tuần.",
             metadata={"source": "faq", "muc": "D1"}),
    Document(page_content="Đồ án chuyên ngành thực hiện theo nhóm từ 2 đến 4 sinh "
             "viên, điểm gồm 60% điểm nhóm và 40% điểm cá nhân.",
             metadata={"source": "faq", "muc": "D2"}),
]

vector_store = InMemoryVectorStore(embeddings)
ids = vector_store.add_documents(documents=KHO_TAI_LIEU)
print(f"Đã index {len(ids)} document")
print("3 id đầu:", ids[:3])

Đã index 8 document
3 id đầu: ['a1c9f82f-b32a-40d4-8243-352b315d5de6', '52354483-22bd-4209-b07c-f65ce6cb5aa6', 'b8516465-a088-40dd-a178-525be25136df']


### 4.1. `similarity_search` — đổi `k`

`k` = số document trả về. Đây là tham số **quan trọng nhất** và cũng hay bị đặt
sai nhất:
- `k` quá nhỏ → thiếu thông tin, model trả lời cụt.
- `k` quá lớn → nhồi rác vào prompt, model bị nhiễu và tốn token.

In [15]:
CAU_HOI = "Sinh viên được đăng ký tối đa bao nhiêu tín chỉ một học kỳ?"

for k in [1, 3, 5]:
    kq = vector_store.similarity_search(CAU_HOI, k=k)
    rule(f"k = {k}  →  {len(kq)} kết quả")
    for i, d in enumerate(kq, 1):
        print(f"  {i}. [{d.metadata.get('source')}] {d.page_content[:65]}...")


k = 1  →  1 kết quả
  1. [quy_che] Mỗi học kỳ chính, sinh viên phải đăng ký tối thiểu 14 tín chỉ và ...

k = 3  →  3 kết quả
  1. [quy_che] Mỗi học kỳ chính, sinh viên phải đăng ký tối thiểu 14 tín chỉ và ...
  2. [faq] Đồ án chuyên ngành thực hiện theo nhóm từ 2 đến 4 sinh viên, điểm...
  3. [quy_che] Từ tuần thứ 3 đến hết tuần thứ 8, sinh viên có thể rút học phần v...

k = 5  →  5 kết quả
  1. [quy_che] Mỗi học kỳ chính, sinh viên phải đăng ký tối thiểu 14 tín chỉ và ...
  2. [faq] Đồ án chuyên ngành thực hiện theo nhóm từ 2 đến 4 sinh viên, điểm...
  3. [quy_che] Từ tuần thứ 3 đến hết tuần thứ 8, sinh viên có thể rút học phần v...
  4. [quy_che] Sinh viên vắng mặt quá 20% số tiết sẽ bị cấm thi và nhận điểm F c...
  5. [quy_che] Sinh viên thuộc diện cảnh báo học vụ chỉ được đăng ký tối đa 14 t...


### 4.2. `similarity_search_with_score` — kèm điểm số

**Cảnh báo quan trọng:** ý nghĩa của `score` **khác nhau tùy provider**.
- `InMemoryVectorStore`, Chroma: thường là **khoảng cách** → nhỏ hơn là gần hơn.
- Một số store khác trả **độ tương đồng** → lớn hơn là gần hơn.

Luôn kiểm tra tài liệu của store đang dùng trước khi đặt ngưỡng lọc.

In [16]:
kq = vector_store.similarity_search_with_score(CAU_HOI, k=4)
print(f"Câu hỏi: {CAU_HOI}\n")
for doc, score in kq:
    print(f"  score={score:.4f}  [{doc.metadata.get('source')}] "
          f"{doc.page_content[:55]}...")

Câu hỏi: Sinh viên được đăng ký tối đa bao nhiêu tín chỉ một học kỳ?

  score=0.0834  [quy_che] Mỗi học kỳ chính, sinh viên phải đăng ký tối thiểu 14 t...
  score=0.0771  [faq] Đồ án chuyên ngành thực hiện theo nhóm từ 2 đến 4 sinh ...
  score=0.0707  [quy_che] Từ tuần thứ 3 đến hết tuần thứ 8, sinh viên có thể rút ...
  score=0.0631  [quy_che] Sinh viên vắng mặt quá 20% số tiết sẽ bị cấm thi và nhậ...


### 4.3. `filter` — lọc theo metadata

Đây là lý do phải đặt metadata cẩn thận từ bước 1. Ví dụ: chỉ tìm trong quy chế,
bỏ qua FAQ.

In [17]:
for dieu_kien, mo_ta in [
    (None,                                          "không lọc"),
    (lambda d: d.metadata.get("source") == "quy_che", "chỉ quy chế"),
    (lambda d: d.metadata.get("source") == "faq",     "chỉ FAQ"),
    (lambda d: d.metadata.get("dieu") == 4,           "chỉ Điều 4"),
]:
    kq = vector_store.similarity_search(CAU_HOI, k=3, filter=dieu_kien)
    rule(f"filter: {mo_ta}  →  {len(kq)} kết quả")
    for d in kq:
        print(f"  [{d.metadata}] {d.page_content[:55]}...")


filter: không lọc  →  3 kết quả
  [{'source': 'quy_che', 'dieu': 4}] Mỗi học kỳ chính, sinh viên phải đăng ký tối thiểu 14 t...
  [{'source': 'faq', 'muc': 'D2'}] Đồ án chuyên ngành thực hiện theo nhóm từ 2 đến 4 sinh ...
  [{'source': 'quy_che', 'dieu': 5}] Từ tuần thứ 3 đến hết tuần thứ 8, sinh viên có thể rút ...

filter: chỉ quy chế  →  3 kết quả
  [{'source': 'quy_che', 'dieu': 4}] Mỗi học kỳ chính, sinh viên phải đăng ký tối thiểu 14 t...
  [{'source': 'quy_che', 'dieu': 5}] Từ tuần thứ 3 đến hết tuần thứ 8, sinh viên có thể rút ...
  [{'source': 'quy_che', 'dieu': 7}] Sinh viên vắng mặt quá 20% số tiết sẽ bị cấm thi và nhậ...

filter: chỉ FAQ  →  3 kết quả
  [{'source': 'faq', 'muc': 'D2'}] Đồ án chuyên ngành thực hiện theo nhóm từ 2 đến 4 sinh ...
  [{'source': 'faq', 'muc': 'B2'}] Đơn phúc khảo nộp tại phòng Khảo thí trong vòng 10 ngày...
  [{'source': 'faq', 'muc': 'D1'}] Sinh viên đã tích lũy tối thiểu 90 tín chỉ được đăng ký...

filter: chỉ Điều 4  →  3 kết quả
  [{'source

### 4.4. MMR — `max_marginal_relevance_search`

**Vấn đề của similarity thuần:** top-k thường là k document **gần giống nhau**,
lặp đi lặp lại cùng một ý.

**MMR** (Maximal Marginal Relevance) cân bằng giữa *liên quan tới câu hỏi* và
*khác biệt với những gì đã chọn*:

$$\text{MMR} = \arg\max_{d \in R \setminus S}\Big[\lambda \cdot \text{sim}(d, q) - (1-\lambda)\max_{s \in S}\text{sim}(d, s)\Big]$$

| Tham số | Ý nghĩa |
|---|---|
| `k` | Số document trả về cuối cùng |
| `fetch_k` | Số ứng viên lấy ra trước khi lọc đa dạng (thường 3–5 lần `k`) |
| `lambda_mult` | 1.0 = chỉ quan tâm liên quan (≡ similarity); 0.0 = chỉ quan tâm đa dạng |

In [18]:
for lam in [1.0, 0.5, 0.0]:
    kq = vector_store.max_marginal_relevance_search(
        CAU_HOI, k=3, fetch_k=8, lambda_mult=lam
    )
    rule(f"lambda_mult = {lam}  "
         f"({'thuần liên quan' if lam==1.0 else 'thuần đa dạng' if lam==0.0 else 'cân bằng'})")
    for i, d in enumerate(kq, 1):
        print(f"  {i}. {d.page_content[:65]}...")


lambda_mult = 1.0  (thuần liên quan)
  1. Mỗi học kỳ chính, sinh viên phải đăng ký tối thiểu 14 tín chỉ và ...
  2. Đồ án chuyên ngành thực hiện theo nhóm từ 2 đến 4 sinh viên, điểm...
  3. Từ tuần thứ 3 đến hết tuần thứ 8, sinh viên có thể rút học phần v...

lambda_mult = 0.5  (cân bằng)
  1. Mỗi học kỳ chính, sinh viên phải đăng ký tối thiểu 14 tín chỉ và ...
  2. Sinh viên thuộc diện cảnh báo học vụ chỉ được đăng ký tối đa 14 t...
  3. Đồ án chuyên ngành thực hiện theo nhóm từ 2 đến 4 sinh viên, điểm...

lambda_mult = 0.0  (thuần đa dạng)
  1. Mỗi học kỳ chính, sinh viên phải đăng ký tối thiểu 14 tín chỉ và ...
  2. Sinh viên thuộc diện cảnh báo học vụ chỉ được đăng ký tối đa 14 t...
  3. Đồ án chuyên ngành thực hiện theo nhóm từ 2 đến 4 sinh viên, điểm...


**Dùng MMR khi nào?** Khi câu hỏi rộng và bạn muốn model thấy nhiều khía cạnh
("tóm tắt các quy định về đăng ký học phần"). Câu hỏi hẹp, tra một con số cụ thể
thì similarity thuần tốt hơn.

---
## 5. Retriever — biến vector store thành `Runnable`

`VectorStore` **không** phải `Runnable`, nên không ghép được vào chain bằng `|`.
`as_retriever()` bọc nó lại thành `Runnable` để có `invoke()`, `batch()`,
`stream()`, và ghép chain được.

| Tham số `as_retriever` | Giá trị |
|---|---|
| `search_type` | `"similarity"` (mặc định) / `"mmr"` / `"similarity_score_threshold"` |
| `search_kwargs` | dict truyền xuống method tương ứng: `k`, `fetch_k`, `lambda_mult`, `filter`, `score_threshold` |

In [19]:
CAU_HINH_RETRIEVER = [
    ({"search_type": "similarity", "search_kwargs": {"k": 2}},
     "similarity, k=2"),
    ({"search_type": "mmr",
      "search_kwargs": {"k": 3, "fetch_k": 8, "lambda_mult": 0.5}},
     "mmr, k=3, cân bằng"),
    ({"search_type": "similarity",
      "search_kwargs": {"k": 3, "filter": lambda d: d.metadata.get("source") == "faq"}},
     "similarity + lọc FAQ"),
]

for cau_hinh, mo_ta in CAU_HINH_RETRIEVER:
    r = vector_store.as_retriever(**cau_hinh)
    kq = r.invoke(CAU_HOI)
    rule(f"{mo_ta}  →  {len(kq)} kết quả")
    for d in kq:
        print(f"  [{d.metadata.get('source')}] {d.page_content[:60]}...")


similarity, k=2  →  2 kết quả
  [quy_che] Mỗi học kỳ chính, sinh viên phải đăng ký tối thiểu 14 tín ch...
  [faq] Đồ án chuyên ngành thực hiện theo nhóm từ 2 đến 4 sinh viên,...

mmr, k=3, cân bằng  →  3 kết quả
  [quy_che] Mỗi học kỳ chính, sinh viên phải đăng ký tối thiểu 14 tín ch...
  [quy_che] Sinh viên thuộc diện cảnh báo học vụ chỉ được đăng ký tối đa...
  [faq] Đồ án chuyên ngành thực hiện theo nhóm từ 2 đến 4 sinh viên,...

similarity + lọc FAQ  →  3 kết quả
  [faq] Đồ án chuyên ngành thực hiện theo nhóm từ 2 đến 4 sinh viên,...
  [faq] Đơn phúc khảo nộp tại phòng Khảo thí trong vòng 10 ngày kể t...
  [faq] Sinh viên đã tích lũy tối thiểu 90 tín chỉ được đăng ký thực...


### 5.1. Lọc theo ngưỡng — chỉ lấy tài liệu đủ liên quan

Mục tiêu: khi câu hỏi nằm **ngoài phạm vi** kho tài liệu, retriever nên trả về
**rỗng** thay vì lôi về document gần nhất (mà vẫn chẳng liên quan gì). Đây là
tuyến phòng thủ đầu tiên chống hallucination.

**Hai cách làm:**

1. `search_type="similarity_score_threshold"` — gọn, nhưng **chỉ chạy trên store
   có cài đặt hàm quy đổi điểm liên quan** (Chroma, PGVector, Qdrant, Pinecone...).
   `InMemoryVectorStore` **không** hỗ trợ, gọi sẽ ra `NotImplementedError`.
2. Tự lọc từ `similarity_search_with_score` — dài hơn vài dòng nhưng **chạy trên
   mọi store**, và bạn thấy rõ điểm số đang so với ngưỡng nào.

Cell dưới thử cách 1, nếu store không hỗ trợ thì tự chuyển sang cách 2.

In [20]:
# --- Cách 1: dùng search_type có sẵn ---
try:
    r_nguong = vector_store.as_retriever(
        search_type="similarity_score_threshold",
        search_kwargs={"k": 4, "score_threshold": 0.3},
    )
    r_nguong.invoke("thử")
    print("[OK] Store này hỗ trợ similarity_score_threshold.")
    HO_TRO_NGUONG = True
except NotImplementedError:
    print("[!] InMemoryVectorStore KHÔNG hỗ trợ similarity_score_threshold.")
    print("    → Chuyển sang cách 2 (tự lọc). Đổi sang Chroma thì cách 1 chạy được.")
    HO_TRO_NGUONG = False

[!] InMemoryVectorStore KHÔNG hỗ trợ similarity_score_threshold.
    → Chuyển sang cách 2 (tự lọc). Đổi sang Chroma thì cách 1 chạy được.


In [21]:
# --- Cách 2: tự lọc theo ngưỡng — chạy trên MỌI vector store ---
from langchain_core.runnables import RunnableLambda

NGUONG = 0.75   # với InMemoryVectorStore, score là KHOẢNG CÁCH: nhỏ hơn = gần hơn

def tim_co_nguong(query: str, k: int = 4, nguong: float = NGUONG):
    ket_qua = vector_store.similarity_search_with_score(query, k=k)
    giu_lai = [(d, s) for d, s in ket_qua if s <= nguong]
    return ket_qua, giu_lai

for q in [
    "Sinh viên đăng ký tối đa bao nhiêu tín chỉ?",   # trong phạm vi
    "Giá vé máy bay đi Đà Nẵng bao nhiêu?",          # ngoài phạm vi
]:
    tat_ca, giu = tim_co_nguong(q)
    rule(f"Câu hỏi: {q}")
    print(f"  Điểm của {len(tat_ca)} ứng viên: "
          f"{[round(s, 3) for _, s in tat_ca]}")
    if giu:
        print(f"  → {len(giu)} document vượt ngưỡng {NGUONG}:")
        for d, s in giu:
            print(f"      ({s:.3f}) {d.page_content[:55]}...")
    else:
        print(f"  → KHÔNG document nào dưới ngưỡng {NGUONG}. "
              "Ứng dụng nên trả lời 'không tìm thấy thông tin'.")

if not ONLINE:
    print("\n[!] FakeEmbedding cho điểm ngẫu nhiên nên ngưỡng ở đây vô nghĩa.")
    print("    Chạy ONLINE để thấy hai câu hỏi tách bạch rõ về điểm số.")


Câu hỏi: Sinh viên đăng ký tối đa bao nhiêu tín chỉ?
  Điểm của 4 ứng viên: [0.1, 0.074, 0.058, 0.012]
  → 4 document vượt ngưỡng 0.75:
      (0.100) Từ tuần thứ 3 đến hết tuần thứ 8, sinh viên có thể rút ...
      (0.074) Mỗi học kỳ chính, sinh viên phải đăng ký tối thiểu 14 t...
      (0.058) Trong học kỳ hè, số tín chỉ tối đa được đăng ký là 12....
      (0.012) Đơn phúc khảo nộp tại phòng Khảo thí trong vòng 10 ngày...

Câu hỏi: Giá vé máy bay đi Đà Nẵng bao nhiêu?
  Điểm của 4 ứng viên: [0.037, 0.006, -0.01, -0.011]
  → 4 document vượt ngưỡng 0.75:
      (0.037) Trong học kỳ hè, số tín chỉ tối đa được đăng ký là 12....
      (0.006) Đồ án chuyên ngành thực hiện theo nhóm từ 2 đến 4 sinh ...
      (-0.010) Mỗi học kỳ chính, sinh viên phải đăng ký tối thiểu 14 t...
      (-0.011) Từ tuần thứ 3 đến hết tuần thứ 8, sinh viên có thể rút ...

[!] FakeEmbedding cho điểm ngẫu nhiên nên ngưỡng ở đây vô nghĩa.
    Chạy ONLINE để thấy hai câu hỏi tách bạch rõ về điểm số.


> **Chọn ngưỡng thế nào?** Không có con số đúng sẵn. Cách làm: chạy bộ câu hỏi
> kiểm thử (`data/bo_cau_hoi_kiem_thu.csv`), in điểm của câu **trong phạm vi** và
> câu **ngoài phạm vi**, rồi chọn ngưỡng nằm giữa hai cụm. Đây chính là bài tập 2
> ở cuối notebook.

### 5.2. `batch()` — xử lý nhiều query song song

Vì retriever là `Runnable`, nó có sẵn `batch()`. Dùng khi cần đánh giá pipeline
trên cả bộ câu hỏi kiểm thử.

In [22]:
retriever = vector_store.as_retriever(search_kwargs={"k": 1})

cac_cau_hoi = [
    "Tối đa bao nhiêu tín chỉ một học kỳ?",
    "Phúc khảo mất bao nhiêu tiền?",
    "Đồ án làm nhóm mấy người?",
    "Vắng bao nhiêu buổi thì cấm thi?",
]

ket_qua_batch = retriever.batch(cac_cau_hoi)

for q, docs in zip(cac_cau_hoi, ket_qua_batch):
    print(f"Q: {q}")
    print(f"A: {docs[0].page_content[:70]}...\n")

Q: Tối đa bao nhiêu tín chỉ một học kỳ?
A: Sinh viên đã tích lũy tối thiểu 90 tín chỉ được đăng ký thực tập doanh...

Q: Phúc khảo mất bao nhiêu tiền?
A: Sinh viên đã tích lũy tối thiểu 90 tín chỉ được đăng ký thực tập doanh...

Q: Đồ án làm nhóm mấy người?
A: Sinh viên thuộc diện cảnh báo học vụ chỉ được đăng ký tối đa 14 tín ch...

Q: Vắng bao nhiêu buổi thì cấm thi?
A: Sinh viên thuộc diện cảnh báo học vụ chỉ được đăng ký tối đa 14 tín ch...



---
## 6. Prompt Template — prompt tái sử dụng được

### `ChatPromptTemplate.from_template()`
Tạo nhanh từ một chuỗi có placeholder `{ten_bien}`.

| Method | Input | Output |
|---|---|---|
| `from_template(str)` | chuỗi có `{var}` | `ChatPromptTemplate` |
| `from_messages(list)` | list các tuple `(role, content)` | `ChatPromptTemplate` |
| `format_messages(**kw)` | giá trị các biến | `list[BaseMessage]` |
| `invoke(dict)` | dict giá trị | `ChatPromptValue` |
| `partial(**kw)` | điền trước một phần | `ChatPromptTemplate` mới |
| `.input_variables` | — | `list[str]` các biến còn thiếu |

In [23]:
from langchain_core.prompts import ChatPromptTemplate

prompt_don_gian = ChatPromptTemplate.from_template(
    "Giải thích khái niệm {khai_niem} cho sinh viên năm {nam} bằng {so_cau} câu."
)

print("Biến cần điền:", prompt_don_gian.input_variables)

msgs = prompt_don_gian.format_messages(khai_niem="tín chỉ", nam=1, so_cau=2)
for m in msgs:
    print(f"\n[{m.type}] {m.content}")

Biến cần điền: ['khai_niem', 'nam', 'so_cau']

[human] Giải thích khái niệm tín chỉ cho sinh viên năm 1 bằng 2 câu.


### 6.1. `from_messages` — có system prompt và lịch sử hội thoại

In [24]:
from langchain_core.prompts import MessagesPlaceholder

prompt_chat = ChatPromptTemplate.from_messages([
    ("system", "Bạn là trợ lý học vụ của trường. Trả lời ngắn gọn, "
               "chỉ dựa trên quy chế. Nếu không biết thì nói không biết."),
    MessagesPlaceholder("lich_su"),        # chỗ cắm lịch sử hội thoại
    ("human", "{cau_hoi}"),
])

print("Biến cần điền:", prompt_chat.input_variables)

from langchain_core.messages import HumanMessage, AIMessage

msgs = prompt_chat.format_messages(
    lich_su=[
        HumanMessage("Em chào thầy"),
        AIMessage("Chào em, thầy có thể giúp gì?"),
    ],
    cau_hoi="Em rút môn ở tuần 10 được không?",
)
for m in msgs:
    print(f"[{m.type:9}] {m.content}")

Biến cần điền: ['cau_hoi', 'lich_su']
[system   ] Bạn là trợ lý học vụ của trường. Trả lời ngắn gọn, chỉ dựa trên quy chế. Nếu không biết thì nói không biết.
[human    ] Em chào thầy
[ai       ] Chào em, thầy có thể giúp gì?
[human    ] Em rút môn ở tuần 10 được không?


### 6.2. `partial()` — điền trước biến cố định

Hữu ích khi một biến luôn giống nhau (ngày hôm nay, tên trường, ngôn ngữ).

In [25]:
from datetime import date

goc = ChatPromptTemplate.from_template(
    "Hôm nay là {ngay}. Trường {truong}. Câu hỏi của sinh viên: {cau_hoi}"
)
print("Trước partial:", goc.input_variables)

da_dien = goc.partial(ngay=str(date.today()), truong="Đại học Hoa Sen")
print("Sau  partial:", da_dien.input_variables)

print()
print(da_dien.format_messages(cau_hoi="Khi nào hết hạn đăng ký?")[0].content)

Trước partial: ['cau_hoi', 'ngay', 'truong']
Sau  partial: ['cau_hoi']

Hôm nay là 2026-10-12. Trường Đại học Hoa Sen. Câu hỏi của sinh viên: Khi nào hết hạn đăng ký?


### 6.3. So sánh chất lượng prompt — 3 mức

Prompt engineering không phải phép thuật, nó là **viết rõ yêu cầu**. Ba mức dưới
đây cho cùng một nhiệm vụ nhưng ràng buộc tăng dần.

In [26]:
MUC_PROMPT = {
    "Mức 1 — mơ hồ":
        "Trả lời câu hỏi: {cau_hoi}",

    "Mức 2 — có ngữ cảnh":
        "Dựa vào thông tin sau:\n{context}\n\nTrả lời câu hỏi: {cau_hoi}",

    "Mức 3 — có ràng buộc đầy đủ":
        "Bạn là trợ lý học vụ. Chỉ dùng THÔNG TIN dưới đây để trả lời.\n"
        "Nếu thông tin không đủ, trả lời đúng câu: "
        "'Tôi không tìm thấy thông tin này trong quy chế.'\n"
        "Trả lời tối đa 3 câu, trích rõ số Điều.\n\n"
        "THÔNG TIN:\n{context}\n\nCÂU HỎI: {cau_hoi}\n\nTRẢ LỜI:",
}

for ten, mau in MUC_PROMPT.items():
    p = ChatPromptTemplate.from_template(mau)
    print(f"{ten}")
    print(f"  biến: {p.input_variables}")
    print(f"  độ dài template: {len(mau)} ký tự")
    print()

Mức 1 — mơ hồ
  biến: ['cau_hoi']
  độ dài template: 26 ký tự

Mức 2 — có ngữ cảnh
  biến: ['cau_hoi', 'context']
  độ dài template: 60 ký tự

Mức 3 — có ràng buộc đầy đủ
  biến: ['cau_hoi', 'context']
  độ dài template: 245 ký tự



**Điểm mấu chốt ở Mức 3:** câu *"Nếu thông tin không đủ, trả lời đúng câu..."* là
thứ giảm hallucination mạnh nhất trong RAG. Không có nó, model sẽ **bịa** khi
retriever lôi về tài liệu không liên quan.

---
## 7. Chat Model — `init_chat_model` và 3 method gọi

| Method | Trả về | Dùng khi |
|---|---|---|
| `invoke(input)` | `AIMessage` | Cần câu trả lời hoàn chỉnh |
| `stream(input)` | iterator `AIMessageChunk` | Hiển thị dần cho người dùng |
| `batch(list)` | `list[AIMessage]` | Nhiều request độc lập, chạy song song |
| `batch_as_completed(list)` | iterator | Như batch nhưng lấy kết quả ngay khi xong |

### Tham số khởi tạo quan trọng

| Tham số | Mặc định | Ý nghĩa |
|---|---|---|
| `model` | — | Tên model, dạng `"provider:ten_model"` |
| `temperature` | tùy provider | 0 = ổn định/lặp lại được; cao = sáng tạo/ngẫu nhiên |
| `max_tokens` | — | Giới hạn độ dài câu trả lời |
| `timeout` | — | Số giây chờ tối đa |
| `max_retries` | 6 | Số lần thử lại khi lỗi mạng / rate limit |

In [27]:
if ONLINE:
    from langchain.chat_models import init_chat_model
    model = init_chat_model(MODEL_NAME, temperature=0, max_tokens=512)
    print("Model:", MODEL_NAME)
else:
    model = None
    print("[OFFLINE] Các cell mục 7–11 sẽ bỏ qua phần gọi LLM.")

[OFFLINE] Các cell mục 7–11 sẽ bỏ qua phần gọi LLM.


In [28]:
@skip_if_offline
def demo_invoke():
    resp = model.invoke("Giải thích 'tín chỉ' trong 2 câu, dành cho sinh viên năm nhất.")
    print("Kiểu trả về:", type(resp).__name__)
    print("\nNội dung:\n", resp.text)
    print("\nToken đã dùng:", resp.usage_metadata)

demo_invoke()

[BỎ QUA] demo_invoke — cần API key. Đặt PROVIDER ở cell 0.1 để chạy.


### 7.1. `temperature` — cùng một prompt, kết quả khác nhau

In [29]:
@skip_if_offline
def demo_temperature():
    from langchain.chat_models import init_chat_model
    cau = "Đặt một tên gọi ngắn gọn cho ứng dụng hỏi đáp quy chế sinh viên."

    for temp in [0.0, 0.0, 1.2, 1.2]:
        m = init_chat_model(MODEL_NAME, temperature=temp, max_tokens=60)
        print(f"temperature={temp}: {m.invoke(cau).text.strip()[:80]}")

demo_temperature()

[BỎ QUA] demo_temperature — cần API key. Đặt PROVIDER ở cell 0.1 để chạy.


Hai lần `temperature=0` cho kết quả **giống hoặc gần giống nhau**; hai lần
`temperature=1.2` cho kết quả **khác nhau rõ rệt**.

**Quy tắc chọn:**
- Trích xuất dữ liệu, phân loại, RAG có căn cứ → `temperature=0`.
- Brainstorm, viết quảng cáo, đặt tên → `temperature` cao.

### 7.2. `stream()` — trả kết quả theo từng chunk

In [30]:
@skip_if_offline
def demo_stream():
    print("Streaming: ", end="")
    full = None
    for chunk in model.stream("Liệt kê 3 điều kiện tốt nghiệp, mỗi điều 1 dòng."):
        print(chunk.text, end="", flush=True)
        full = chunk if full is None else full + chunk
    print(f"\n\n→ Ghép lại thành {type(full).__name__}, "
          f"tổng {len(full.text)} ký tự")

demo_stream()

[BỎ QUA] demo_stream — cần API key. Đặt PROVIDER ở cell 0.1 để chạy.


Điểm hay: các `AIMessageChunk` **cộng được với nhau** bằng `+`, ghép lại thành
message hoàn chỉnh y như `invoke()` trả về.

### 7.3. `batch()` — nhiều câu hỏi song song

In [31]:
@skip_if_offline
def demo_batch():
    import time
    cau_hoi = [
        "Tín chỉ là gì? (1 câu)",
        "GPA là gì? (1 câu)",
        "Học phần tiên quyết là gì? (1 câu)",
    ]

    t0 = time.time()
    tuan_tu = [model.invoke(q) for q in cau_hoi]
    t_tuan_tu = time.time() - t0

    t0 = time.time()
    song_song = model.batch(cau_hoi, config={"max_concurrency": 3})
    t_song_song = time.time() - t0

    print(f"Tuần tự  : {t_tuan_tu:.2f}s")
    print(f"batch()  : {t_song_song:.2f}s   "
          f"→ nhanh hơn {t_tuan_tu/t_song_song:.1f} lần\n")
    for q, r in zip(cau_hoi, song_song):
        print(f"Q: {q}\nA: {r.text.strip()[:90]}\n")

demo_batch()

[BỎ QUA] demo_batch — cần API key. Đặt PROVIDER ở cell 0.1 để chạy.


---
## 8. Output Parser — ép định dạng đầu ra

### `StrOutputParser`
Chuyển `AIMessage` → `str`. Nhỏ nhưng dùng gần như mọi chain, vì bước sau
(in ra, nối chuỗi) cần `str` chứ không cần cả object message.

### `with_structured_output(schema)`
Ép model trả về **đúng schema** (Pydantic / TypedDict / JSON Schema). Đây là cách
đúng để lấy dữ liệu có cấu trúc — **không** parse JSON thủ công từ text, vì model
hay kèm thêm ```json fence hoặc lời dẫn.

In [32]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

from langchain_core.messages import AIMessage
gia_lap = AIMessage(content="Tín chỉ là đơn vị đo khối lượng học tập.")

print("Trước parser:", type(gia_lap).__name__, "→", repr(gia_lap)[:60], "...")
print("Sau   parser:", type(parser.invoke(gia_lap)).__name__, "→",
      repr(parser.invoke(gia_lap)))

Trước parser: AIMessage → AIMessage(content='Tín chỉ là đơn vị đo khối lượng học tập.' ...
Sau   parser: TextAccessor → 'Tín chỉ là đơn vị đo khối lượng học tập.'


In [33]:
from pydantic import BaseModel, Field

class ThongTinDieuKhoan(BaseModel):
    '''Thông tin trích xuất từ một điều khoản quy chế.'''
    so_dieu: int = Field(description="Số của điều khoản")
    tieu_de: str = Field(description="Tiêu đề điều khoản")
    cac_con_so: list[int] = Field(description="Mọi con số định lượng xuất hiện")
    tom_tat: str = Field(description="Tóm tắt trong đúng 1 câu")


@skip_if_offline
def demo_structured():
    model_co_cau_truc = model.with_structured_output(ThongTinDieuKhoan)
    kq = model_co_cau_truc.invoke(
        "Trích xuất thông tin từ điều khoản sau:\n\n"
        "Điều 4. Đăng ký học phần. Mỗi học kỳ chính, sinh viên phải đăng ký "
        "tối thiểu 14 tín chỉ và tối đa 24 tín chỉ. Học kỳ hè tối đa 12 tín chỉ."
    )
    print("Kiểu trả về:", type(kq).__name__)
    print("  so_dieu    :", kq.so_dieu)
    print("  tieu_de    :", kq.tieu_de)
    print("  cac_con_so :", kq.cac_con_so)
    print("  tom_tat    :", kq.tom_tat)
    print("\nDict:", kq.model_dump())

demo_structured()

[BỎ QUA] demo_structured — cần API key. Đặt PROVIDER ở cell 0.1 để chạy.


> **Vì sao không tự parse JSON?** Vì `with_structured_output` dùng cơ chế
> *function calling* hoặc *native structured output* của provider — model bị ép ở
> tầng API phải sinh đúng schema, không phải "xin" nó trả JSON rồi hy vọng.
> Pydantic còn **validate** kiểu dữ liệu giúp bạn.

---
## 9. LCEL — LangChain Expression Language

Đây là **phần quan trọng nhất** của LangChain hiện đại. Hiểu nó là hiểu 80% framework.

### Ý tưởng
Mọi thành phần (prompt, model, parser, retriever, cả hàm Python thường) đều là
`Runnable`. `Runnable` nào cũng có `invoke` / `stream` / `batch` / `ainvoke`.
Toán tử `|` nối chúng lại: **output của cái trước là input của cái sau**.

```
prompt | model | parser
  dict → PromptValue → AIMessage → str
```

### Các Runnable phụ trợ

| Runnable | Tác dụng |
|---|---|
| `RunnablePassthrough()` | Cho input đi thẳng qua, không đổi |
| `RunnableParallel({...})` | Chạy nhiều nhánh song song, gom vào 1 dict |
| `RunnableLambda(f)` | Bọc một hàm Python thường thành Runnable |
| `.assign(k=...)` | Thêm khóa mới vào dict đang chảy qua |

In [34]:
from langchain_core.runnables import (
    RunnablePassthrough, RunnableParallel, RunnableLambda
)

# --- RunnableLambda: hàm Python thường trở thành Runnable ---
dem_tu = RunnableLambda(lambda s: len(s.split()))
viet_hoa = RunnableLambda(lambda s: s.upper())

print("dem_tu.invoke  :", dem_tu.invoke("một hai ba bốn năm"))
print("viet_hoa.invoke:", viet_hoa.invoke("langchain"))

# --- Nối bằng | ---
chuoi = RunnableLambda(lambda s: s.strip()) | viet_hoa | RunnableLambda(lambda s: s[::-1])
print("chuỗi 3 bước   :", chuoi.invoke("  langchain  "))

dem_tu.invoke  : 5
viet_hoa.invoke: LANGCHAIN
chuỗi 3 bước   : NIAHCGNAL


In [35]:
# --- RunnableParallel: chạy nhiều nhánh cùng lúc ---
phan_tich = RunnableParallel({
    "goc":      RunnablePassthrough(),
    "so_tu":    RunnableLambda(lambda s: len(s.split())),
    "so_ky_tu": RunnableLambda(lambda s: len(s)),
    "hoa":      RunnableLambda(lambda s: s.upper()),
})

import json
kq = phan_tich.invoke("Sinh viên đăng ký tối đa 24 tín chỉ")
print(json.dumps(kq, ensure_ascii=False, indent=2))

{
  "goc": "Sinh viên đăng ký tối đa 24 tín chỉ",
  "so_tu": 9,
  "so_ky_tu": 35,
  "hoa": "SINH VIÊN ĐĂNG KÝ TỐI ĐA 24 TÍN CHỈ"
}


In [36]:
# --- .assign(): thêm khóa vào dict đang chảy qua chain ---
them_thong_tin = (
    RunnablePassthrough.assign(do_dai=lambda x: len(x["cau_hoi"]))
    .assign(co_dau_hoi=lambda x: "?" in x["cau_hoi"])
)

print(them_thong_tin.invoke({"cau_hoi": "Tối đa bao nhiêu tín chỉ?"}))

{'cau_hoi': 'Tối đa bao nhiêu tín chỉ?', 'do_dai': 25, 'co_dau_hoi': True}


### 9.1. Mẫu chuẩn của RAG chain

Đây là "khuôn" mà gần như mọi RAG chain đều dùng:

```python
{"context": retriever | format_docs, "cau_hoi": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
```

Đọc từ trên xuống: một chuỗi `str` (câu hỏi) đi vào; nhánh `context` đưa nó qua
retriever rồi gộp thành text; nhánh `cau_hoi` giữ nguyên; hai nhánh gom thành
dict đúng hai biến mà prompt cần.

In [37]:
def format_docs(docs):
    '''Gộp list[Document] thành một chuỗi có đánh số nguồn.'''
    return "\n\n".join(
        f"[Nguồn {i}: {d.metadata.get('source','?')} "
        f"- {d.metadata.get('dieu') or d.metadata.get('muc','?')}]\n{d.page_content}"
        for i, d in enumerate(docs, 1)
    )

# Chạy được offline — chỉ là retriever + hàm Python
nhanh_context = vector_store.as_retriever(search_kwargs={"k": 2}) | format_docs
print(nhanh_context.invoke("Sinh viên đăng ký tối đa bao nhiêu tín chỉ?"))

[Nguồn 1: quy_che - 5]
Từ tuần thứ 3 đến hết tuần thứ 8, sinh viên có thể rút học phần và được ghi ký hiệu W trên bảng điểm.

[Nguồn 2: quy_che - 4]
Mỗi học kỳ chính, sinh viên phải đăng ký tối thiểu 14 tín chỉ và tối đa 24 tín chỉ.


---
## 10. RAG hoàn chỉnh — ghép tất cả lại

Pipeline: **câu hỏi → retriever → format → prompt → model → parser → câu trả lời**

In [38]:
PROMPT_RAG = ChatPromptTemplate.from_template(
    "Bạn là trợ lý học vụ. CHỈ dùng THÔNG TIN dưới đây để trả lời.\n"
    "Nếu thông tin không đủ, trả lời đúng câu: "
    "'Tôi không tìm thấy thông tin này trong quy chế.'\n"
    "Trả lời tối đa 3 câu và ghi rõ nguồn.\n\n"
    "THÔNG TIN:\n{context}\n\n"
    "CÂU HỎI: {cau_hoi}\n\n"
    "TRẢ LỜI:"
)

@skip_if_offline
def build_rag(k=3, search_type="similarity"):
    r = vector_store.as_retriever(search_type=search_type, search_kwargs={"k": k})
    return (
        {"context": r | format_docs, "cau_hoi": RunnablePassthrough()}
        | PROMPT_RAG
        | model
        | StrOutputParser()
    )

rag = build_rag()
print("Chain đã dựng." if rag else "")

[BỎ QUA] build_rag — cần API key. Đặt PROVIDER ở cell 0.1 để chạy.



In [39]:
@skip_if_offline
def chay_rag():
    cau_hoi_test = [
        "Sinh viên được đăng ký tối đa bao nhiêu tín chỉ?",
        "Phúc khảo mất bao nhiêu tiền và nộp ở đâu?",
        "Đồ án chuyên ngành làm nhóm mấy người?",
        "Học phí một tín chỉ là bao nhiêu?",   # <- CÂU BẪY, không có trong data
    ]
    for q in cau_hoi_test:
        rule(f"HỎI: {q}")
        print(rag.invoke(q))

chay_rag()

[BỎ QUA] chay_rag — cần API key. Đặt PROVIDER ở cell 0.1 để chạy.


### 10.1. Câu bẫy — kiểm tra hallucination

Câu hỏi cuối (*"Học phí một tín chỉ là bao nhiêu?"*) **không có** trong kho tài
liệu. Một RAG tốt phải trả lời *"không tìm thấy thông tin"*, chứ không được bịa
một con số.

**Đây là tiêu chí chấm quan trọng nhất khi đánh giá RAG.** Nhóm nào demo mà model
bịa số ở câu này là pipeline chưa đạt.

Hai cách siết:
1. Câu lệnh "nếu không đủ thì nói không biết" trong prompt (đã làm ở trên).
2. `search_type="similarity_score_threshold"` để retriever trả rỗng khi không đủ
   liên quan (mục 5.1).

### 10.2. Ảnh hưởng của `k` tới câu trả lời

In [40]:
@skip_if_offline
def so_sanh_k():
    q = "Các quy định về số tín chỉ được đăng ký?"
    for k in [1, 3, 6]:
        rule(f"k = {k}")
        print(build_rag(k=k).invoke(q))

so_sanh_k()

[BỎ QUA] so_sanh_k — cần API key. Đặt PROVIDER ở cell 0.1 để chạy.


### 10.3. Trả về kèm nguồn trích dẫn

Trong thực tế, câu trả lời không kèm nguồn thì người dùng không kiểm chứng được.
Dùng `RunnableParallel` để giữ lại documents song song với câu trả lời.

In [41]:
@skip_if_offline
def rag_co_nguon():
    r = vector_store.as_retriever(search_kwargs={"k": 3})

    chain = RunnableParallel({
        "documents": r,
        "cau_hoi": RunnablePassthrough(),
    }).assign(
        tra_loi=(
            {"context": lambda x: format_docs(x["documents"]),
             "cau_hoi": lambda x: x["cau_hoi"]}
            | PROMPT_RAG | model | StrOutputParser()
        )
    )

    kq = chain.invoke("Vắng bao nhiêu phần trăm số tiết thì bị cấm thi?")
    print("TRẢ LỜI:\n", kq["tra_loi"])
    print("\nNGUỒN THAM CHIẾU:")
    for d in kq["documents"]:
        print(f"  - {d.metadata} :: {d.page_content[:60]}...")

rag_co_nguon()

[BỎ QUA] rag_co_nguon — cần API key. Đặt PROVIDER ở cell 0.1 để chạy.


---
## 11. Agent — model tự quyết định gọi tool

Khác biệt cốt lõi so với chain:

| | Chain (RAG 2 bước) | Agent |
|---|---|---|
| Luồng | Cố định: luôn retrieve rồi generate | Model tự quyết định gọi tool nào, mấy lần |
| Kiểm soát | Cao | Thấp |
| Độ trễ | Dự đoán được | Biến thiên |
| Phù hợp | FAQ, tra cứu tài liệu | Trợ lý nhiều nguồn, cần suy luận nhiều bước |

`create_agent` là API chính của LangChain 1.0, chạy trên runtime LangGraph.

### Tạo tool bằng `@tool`

**Quan trọng:** docstring của hàm chính là **mô tả gửi cho model** để nó quyết
định khi nào gọi. Docstring viết ẩu → model gọi sai tool. Type hint trở thành
schema tham số.

In [42]:
from langchain.tools import tool

@tool
def tra_cuu_quy_che(cau_hoi: str) -> str:
    '''Tra cứu quy chế đào tạo và FAQ sinh viên.

    Dùng tool này cho MỌI câu hỏi về: tín chỉ, đăng ký học phần, rút môn,
    điểm số, thi cử, phúc khảo, học bổng, thực tập, đồ án, tốt nghiệp.

    Args:
        cau_hoi: Câu hỏi của sinh viên, viết đầy đủ bằng tiếng Việt.
    '''
    docs = vector_store.similarity_search(cau_hoi, k=3)
    if not docs:
        return "Không tìm thấy thông tin liên quan trong quy chế."
    return format_docs(docs)


@tool
def tinh_gpa(diem_va_tin_chi: str) -> str:
    '''Tính điểm trung bình tích lũy (GPA) theo thang 4.

    Args:
        diem_va_tin_chi: Chuỗi các cặp 'điểm:số_tín_chỉ' ngăn bởi dấu phẩy.
            Ví dụ: "3.7:3, 3.0:4, 2.5:2"
    '''
    try:
        tong_diem, tong_tc = 0.0, 0
        for cap in diem_va_tin_chi.split(","):
            d, tc = cap.strip().split(":")
            tong_diem += float(d) * int(tc)
            tong_tc += int(tc)
        return (f"GPA = {tong_diem/tong_tc:.2f} "
                f"(tổng {tong_tc} tín chỉ)")
    except Exception as e:
        return f"Lỗi định dạng đầu vào: {e}"


# Tool vẫn gọi trực tiếp được để test — chạy offline
print(tinh_gpa.invoke({"diem_va_tin_chi": "3.7:3, 3.0:4, 2.5:2"}))
print()
print("Schema mà model nhìn thấy:")
print("  name       :", tinh_gpa.name)
print("  description:", tinh_gpa.description[:70], "...")
print("  args       :", tinh_gpa.args)

GPA = 3.12 (tổng 9 tín chỉ)

Schema mà model nhìn thấy:
  name       : tinh_gpa
  description: Tính điểm trung bình tích lũy (GPA) theo thang 4.

Args:
    diem_va_t ...
  args       : {'diem_va_tin_chi': {'title': 'Diem Va Tin Chi', 'type': 'string'}}


In [43]:
@skip_if_offline
def demo_agent():
    from langchain.agents import create_agent

    agent = create_agent(
        model=model,
        tools=[tra_cuu_quy_che, tinh_gpa],
        system_prompt=(
            "Bạn là trợ lý học vụ. Luôn dùng tool tra_cuu_quy_che trước khi trả "
            "lời câu hỏi về quy định. Không được bịa thông tin. "
            "Trả lời ngắn gọn bằng tiếng Việt."
        ),
    )

    cau_hoi = ("Em có 3 môn: 3.7 (3 tín chỉ), 3.0 (4 tín chỉ), 2.5 (2 tín chỉ). "
               "GPA của em là bao nhiêu và em có đủ điều kiện làm khóa luận không?")

    kq = agent.invoke({"messages": [{"role": "user", "content": cau_hoi}]})

    rule("DIỄN BIẾN AGENT")
    for m in kq["messages"]:
        loai = type(m).__name__
        if getattr(m, "tool_calls", None):
            for tc in m.tool_calls:
                print(f"[{loai}] → GỌI TOOL {tc['name']}({tc['args']})")
        elif loai == "ToolMessage":
            print(f"[{loai}] ← {str(m.content)[:100]}...")
        elif m.text:
            print(f"[{loai}] {m.text[:200]}")
        print()

demo_agent()

[BỎ QUA] demo_agent — cần API key. Đặt PROVIDER ở cell 0.1 để chạy.


Câu hỏi này cần **2 tool khác nhau**: `tinh_gpa` để tính, `tra_cuu_quy_che` để
biết ngưỡng GPA làm khóa luận. Agent tự nhận ra và gọi cả hai — điều mà một chain
cố định không làm được.

---
## 12. Bảng tổng kết tham số

### Text Splitter
| Tham số | Giá trị đề xuất | Ghi chú |
|---|---|---|
| `chunk_size` | 500–1000 ký tự | Văn bản hành chính tiếng Việt: 800 |
| `chunk_overlap` | 10–20% của `chunk_size` | 100–200 là phổ biến |
| `separators` | để mặc định | Chỉ đổi khi có cấu trúc đặc thù |
| `add_start_index` | `True` | Gần như luôn nên bật |

### Retriever
| Tình huống | Cấu hình |
|---|---|
| Tra cứu một dữ kiện cụ thể | `similarity`, `k=2..3` |
| Câu hỏi rộng, cần nhiều góc nhìn | `mmr`, `k=4`, `fetch_k=20`, `lambda_mult=0.5` |
| Cần chống trả lời bừa | `similarity_score_threshold`, `score_threshold=0.3..0.5` |
| Có nhiều nguồn tài liệu | thêm `filter` theo metadata |

### Chat Model
| Tác vụ | `temperature` |
|---|---|
| RAG, trích xuất, phân loại | `0` |
| Viết lại câu, tóm tắt | `0.3` |
| Brainstorm, sáng tạo | `0.7–1.2` |

### Sai lầm thường gặp

| Sai lầm | Hậu quả | Cách sửa |
|---|---|---|
| `chunk_overlap=0` | Câu bị cắt đôi, mất ngữ cảnh | Đặt 10–20% `chunk_size` |
| `k` quá lớn (10+) | Prompt nhiễu, tốn token, chất lượng giảm | `k=3` rồi tăng dần |
| Không có metadata | Không trích nguồn, không lọc được | Đặt metadata từ khi load |
| Prompt thiếu "nếu không biết thì nói không biết" | Model bịa | Thêm ràng buộc vào prompt |
| `temperature` cao cho RAG | Câu trả lời không ổn định | Đặt `0` |
| Parse JSON thủ công từ text | Vỡ khi model thêm fence | `with_structured_output()` |
| Dùng `langchain.chains` | `ModuleNotFoundError` ở v1.0 | Dùng LCEL (`\|`) |

---

## 13. Bài tập cho các nhóm

1. **Cơ bản** — Thay 2 file dữ liệu trong `data/` bằng tài liệu của nhóm bạn
   (slide môn học, nội quy CLB, tài liệu API...). Chạy lại toàn bộ notebook.

2. **Trung bình** — Chạy toàn bộ 22 câu trong `data/bo_cau_hoi_kiem_thu.csv` qua
   RAG chain. Lập bảng: câu hỏi / đáp án mong đợi / đáp án model / đúng-sai.
   Tính tỷ lệ chính xác. Đặc biệt kiểm tra 2 câu bẫy Q21, Q22.

3. **Nâng cao** — Chạy lại bài 2 với 3 cấu hình `chunk_size` khác nhau
   (300 / 800 / 1500). Vẽ biểu đồ tỷ lệ chính xác theo `chunk_size`. Giải thích
   tại sao có điểm tối ưu.

4. **Nâng cao** — Thêm một tool thứ ba cho agent (tra cứu lịch thi, tính học phí,
   hoặc bất cứ gì phù hợp dữ liệu nhóm bạn). Chứng minh agent biết chọn đúng tool
   qua ít nhất 3 câu hỏi khác nhau.

5. **Thử thách** — Dựng giao diện Streamlit cho RAG chain, có ô nhập câu hỏi,
   hiển thị câu trả lời kèm nguồn trích dẫn, và slider cho `k`.